In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

import importlib
import module
importlib.reload(module)

In [ ]:
# DOWNLOAD DATA

benchmark_ticker = "AOR"

download_tickers = [
    "SPY",
    "XLE",
    "GLD",
    benchmark_ticker
]

tickers = [
    "SPY_VIX",
    "XLE_OIL",
    "GLD_GOLD",
    benchmark_ticker
]

vix_ticker = "^VIX"
vix3m_ticker = "^VIX3M"
oil_ticker = "CL=F"
bond_ticker = "TLT"

indicator_tickers = [
    vix_ticker,
    vix3m_ticker,
    oil_ticker,
    bond_ticker
]

start_date = "2015-01-01"
end_date = "2025-12-31"

df_downloaded_prices, _ = module.download_stock_price_data(
    download_tickers,
    start_date,
    end_date
)

df_indicator_prices, _ = module.download_stock_price_data(
    indicator_tickers,
    start_date,
    end_date
)

common_index = df_downloaded_prices.index.intersection(
    df_indicator_prices.index
)

df_downloaded_prices = df_downloaded_prices.loc[common_index]
df_indicator_prices = df_indicator_prices.loc[common_index]

df_prices = pd.DataFrame(index=common_index)

df_prices["SPY_VIX"] = df_downloaded_prices["SPY"]
df_prices["XLE_OIL"] = df_downloaded_prices["XLE"]
df_prices["GLD_GOLD"] = df_downloaded_prices["GLD"]
df_prices[benchmark_ticker] = df_downloaded_prices[benchmark_ticker]

df_price_changes = module.price_changes_from_price_frame(df_prices)

print("First common date:", df_prices.index.min())
print("Last common date:", df_prices.index.max())

df_prices.tail()

In [ ]:
# DEFINE THREE TRADING SIGNALS

# Signal 0:
# Trades SPY using VIX term structure and SPY momentum.

# Signal 1:
# Trades XLE using crude oil momentum, XLE trend, XLE momentum,
# and XLE relative strength versus SPY.

# Signal 2:
# Trades GLD using gold trend, gold momentum, and Treasury bond momentum.

Signal 0 is a SPY / VIX term-structure signal.

The strategy trades SPY. The VIX index and the VIX3M index are not traded directly; they are only used as information variables.

The economic idea is that the VIX term structure contains information about the equity-market risk regime. A high VIX3M/VIX ratio usually indicates a more normal volatility term structure, while an inverted or flat structure can indicate short-term stress. The signal also uses SPY momentum to avoid investing when the broad equity market is in a clearly weak trend.

The VIX term-structure ratio is:

$$
R_t^{VIX}
=
\frac{VIX3M_t}{VIX_t}
$$

The SPY momentum condition is:

$$
M_t^{SPY}
=
\frac{SPY_t}{SPY_{t-168}} - 1
>
-0.025
$$

The final signal is:

$$
S_t^{SPY}
=
\begin{cases}
1, & \text{if } R_t^{VIX} > 1.10 \text{ and } M_t^{SPY} > -0.025 \\
0, & \text{otherwise}
\end{cases}
$$

In [ ]:
signal_0_ratio_threshold = 1.10
signal_0_momentum_window = 168
signal_0_momentum_threshold = -0.025


def signal_0(series):

    return module.vix_term_structure_signal(
        target_series=series,
        vix_series=df_indicator_prices[vix_ticker],
        vix3m_series=df_indicator_prices[vix3m_ticker],
        ratio_threshold=signal_0_ratio_threshold,
        momentum_window=signal_0_momentum_window,
        momentum_threshold=signal_0_momentum_threshold
    )

Signal 1 is an oil and energy relative-strength signal.

The strategy trades XLE. Crude oil futures and SPY are not traded directly; they are only used as information variables.

The idea is that energy equities should perform better when the energy sector is in an upward trend, crude oil momentum is supportive, XLE has positive own momentum, and XLE is not underperforming the broad equity market.

The XLE trend condition is:

$$
XLE_t > MA_t^{XLE,200}
$$

The oil momentum condition is:

$$
M_t^{Oil}
=
\frac{Oil_t}{Oil_{t-126}} - 1
>
-0.05
$$

The XLE momentum condition is:

$$
M_t^{XLE}
=
\frac{XLE_t}{XLE_{t-210}} - 1
>
0
$$

The relative-strength ratio is:

$$
RS_t^{XLE/SPY}
=
\frac{XLE_t}{SPY_t}
$$

The relative-strength momentum condition is:

$$
M_t^{RS}
=
\frac{RS_t^{XLE/SPY}}{RS_{t-21}^{XLE/SPY}} - 1
>
-0.025
$$

The final signal is:

$$
S_t^{XLE}
=
\begin{cases}
1, & \text{if all four conditions are satisfied} \\
0, & \text{otherwise}
\end{cases}
$$

In [ ]:
signal_1_trend_window = 200

signal_1_oil_momentum_window = 126
signal_1_oil_momentum_threshold = -0.05

signal_1_target_momentum_window = 210
signal_1_target_momentum_threshold = 0.0

signal_1_relative_strength_window = 21
signal_1_relative_strength_threshold = -0.025


def signal_1(series):

    return module.oil_energy_relative_strength_signal(
        target_series=series,
        oil_series=df_indicator_prices[oil_ticker],
        market_series=df_downloaded_prices["SPY"],
        trend_window=signal_1_trend_window,
        oil_momentum_window=signal_1_oil_momentum_window,
        oil_momentum_threshold=signal_1_oil_momentum_threshold,
        target_momentum_window=signal_1_target_momentum_window,
        target_momentum_threshold=signal_1_target_momentum_threshold,
        relative_strength_window=signal_1_relative_strength_window,
        relative_strength_threshold=signal_1_relative_strength_threshold
    )

Signal 2 is a gold regime signal.

The strategy trades GLD. TLT is not traded directly; it is only used as an information variable for the bond and interest-rate environment.

The economic idea is that gold can provide a different return driver from equities and energy stocks. The signal invests in gold only when GLD is in an upward trend, gold has positive medium-term momentum, and long-term Treasury bonds are not strongly negative.

The GLD trend condition is:

$$
GLD_t > MA_t^{GLD,200}
$$

The GLD momentum condition is:

$$
M_t^{GLD}
=
\frac{GLD_t}{GLD_{t-126}} - 1
>
0
$$

The TLT momentum condition is:

$$
M_t^{TLT}
=
\frac{TLT_t}{TLT_{t-126}} - 1
>
-0.02
$$

The final signal is:

$$
S_t^{GLD}
=
\begin{cases}
1, & \text{if all three conditions are satisfied} \\
0, & \text{otherwise}
\end{cases}
$$

In [ ]:
signal_2_trend_window = 200
signal_2_gold_momentum_window = 126
signal_2_gold_momentum_threshold = 0.0
signal_2_bond_momentum_window = 126
signal_2_bond_momentum_threshold = -0.02


def signal_2(series):

    return module.gold_regime_signal(
        gold_series=series,
        bond_series=df_indicator_prices[bond_ticker],
        trend_window=signal_2_trend_window,
        gold_momentum_window=signal_2_gold_momentum_window,
        gold_momentum_threshold=signal_2_gold_momentum_threshold,
        bond_momentum_window=signal_2_bond_momentum_window,
        bond_momentum_threshold=signal_2_bond_momentum_threshold
    )

In [ ]:
signals = {
    tickers[0]: signal_0(df_prices[tickers[0]]),
    tickers[1]: signal_1(df_prices[tickers[1]]),
    tickers[2]: signal_2(df_prices[tickers[2]])
}

execution_delay_days = 1

for ticker in tickers[:-1]:

    signals[ticker] = module.apply_execution_delay(
        signals[ticker],
        delay_days=execution_delay_days
    )

df_position_open = pd.concat([
    signals[tickers[0]]["signal"].rename(tickers[0]),
    signals[tickers[1]]["signal"].rename(tickers[1]),
    signals[tickers[2]]["signal"].rename(tickers[2])
], axis=1)

df_position_changes = pd.concat([
    signals[tickers[0]]["position_change"].rename(tickers[0]),
    signals[tickers[1]]["position_change"].rename(tickers[1]),
    signals[tickers[2]]["position_change"].rename(tickers[2])
], axis=1)

df_position_open.tail()

In [ ]:
# ALLOCATE CAPITAL AND COMPUTE RESULTING POSITIONS
initial_cash = 1.0
capital_fraction_per_trade = 0.30

# DO NOT MODIFY THIS CELL BELOW THIS LINE
position = []

def open_trades(position, position_change):
    vec = np.maximum([position_change[ticker] for ticker in tickers[:-1]], [0])
    vec = position[-1] * (1 - np.power((1 - capital_fraction_per_trade), np.sum(vec))) * vec / (1 if (np.nansum(vec) == 0.0) else np.nansum(vec))
    return np.append(vec + position[:-1], position[-1] - np.sum(vec))

def hold_trades(position, price_change):
    return np.concatenate((position[:-1] * price_change[:-1], [position[-1]]))

def close_trades(position, position_change):
    vec = np.concatenate((np.array([position_change[ticker] < 0.0 for ticker in tickers[:-1]]), [False]))
    position[-1] = position[-1] + np.sum(position[vec])
    position[vec] = 0.0
    return position
    
is_first = True
for idx, position_change in df_position_changes.iterrows():
    if is_first:
        position.append(open_trades(np.concatenate((np.zeros(len(df_position_changes.columns)), [initial_cash])), position_change))
        is_first = False
    else:
        hlpr_pos = hold_trades(position[-1], df_price_changes.loc[[idx]].to_numpy()[0])
        hlpr_pos = close_trades(hlpr_pos, position_change)
        position.append(open_trades(hlpr_pos, position_change))

df_position = pd.DataFrame(position, index = df_prices.index, columns = tickers[:-1] + ['cash'])

In [ ]:
statistics_rows = {}

portfolio_value = df_position.sum(axis=1).to_numpy(dtype=float)

statistics_rows["Combined Strategy"] = (
    module.performance_statistics_from_values(
        portfolio_value
    )
)

benchmark_values = df_prices[tickers[-1]].to_numpy(dtype=float)

benchmark_curve = (
    benchmark_values
    / benchmark_values[0]
)

statistics_rows["Benchmark"] = (
    module.performance_statistics_from_values(
        benchmark_curve
    )
)

df_statistics = pd.DataFrame(statistics_rows).T

df_statistics = df_statistics[
    [
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Max Drawdown"
    ]
]

df_statistics.round(4)

In [ ]:
# COMPUTE MEANINGFUL PLOTS OF YOUR STRATEGY AND LABEL THEM IN AN UNDERSTANDABLE WAY

portfolio_value = df_position.sum(axis=1).to_numpy(dtype=float)

portfolio_curve = (
    portfolio_value
    / portfolio_value[0]
)

benchmark_values = df_prices[tickers[-1]].to_numpy(dtype=float)

benchmark_curve = (
    benchmark_values
    / benchmark_values[0]
)

plt.figure(figsize=(12, 5))

plt.plot(
    df_prices.index,
    portfolio_curve,
    label="Combined strategy portfolio"
)

plt.plot(
    df_prices.index,
    benchmark_curve,
    label="Benchmark " + tickers[-1],
    linestyle="--"
)

plt.title("Combined Strategy Performance Since 2015")
plt.xlabel("Date")
plt.ylabel("Wealth index")
plt.legend()
plt.grid(True)

plt.show()